# 🔍 Competitor Intelligence v2 — Product Analytics
**Real-time scraping · Scoring engine · A/B Generator**

Pipeline:
```
Scrape real-time → Score 6 tiêu chí → Detect changes → A/B variants
Landing · Pricing · G2 · Meta Ads
```

▶️ Chạy từng cell theo thứ tự

---
## ⚙️ CELL 1 — Cài thư viện

In [ ]:
!pip install requests beautifulsoup4 -q
print('✅ Done')

---
## 💾 CELL 2 — Mount Drive + Setup DB

In [ ]:
from google.colab import drive
import os, sqlite3, json, hashlib
from datetime import date

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/competitor-intel-v2'
os.makedirs(BASE_DIR, exist_ok=True)
DB_PATH = f'{BASE_DIR}/intel.db'

def setup_db():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS snapshots (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            date          TEXT,
            competitor    TEXT,
            headline      TEXT,
            subheadline   TEXT,
            cta           TEXT,
            plans         TEXT,
            prices        TEXT,
            has_free      INTEGER,
            pricing_model TEXT,
            g2_rating     REAL,
            g2_count      INTEGER,
            ad_count      INTEGER,
            ad_messages   TEXT,
            score_total   REAL,
            score_detail  TEXT,
            hashes_json   TEXT
        );
        CREATE TABLE IF NOT EXISTS changes (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            detected_on  TEXT,
            competitor   TEXT,
            field        TEXT,
            value_before TEXT,
            value_after  TEXT
        );
    """)
    conn.close()

setup_db()
print(f'✅ DB ready: {DB_PATH}')

---
## 🏢 CELL 3 — Config competitors
> ✏️ Thêm/bớt competitor tại đây

In [ ]:
COMPETITORS = [
    {
        'name': 'Amplitude',
        'landing_url': 'https://amplitude.com',
        'pricing_url': 'https://amplitude.com/pricing',
        'g2_slug': 'amplitude',
        'meta_page_id': 'amplitudeanalytics'
    },
    {
        'name': 'Mixpanel',
        'landing_url': 'https://mixpanel.com',
        'pricing_url': 'https://mixpanel.com/pricing',
        'g2_slug': 'mixpanel',
        'meta_page_id': 'mixpanel'
    },
    {
        'name': 'PostHog',
        'landing_url': 'https://posthog.com',
        'pricing_url': 'https://posthog.com/pricing',
        'g2_slug': 'posthog',
        'meta_page_id': 'posthog'
    },
    {
        'name': 'Heap',
        'landing_url': 'https://heap.io',
        'pricing_url': 'https://heap.io/pricing',
        'g2_slug': 'heap',
        'meta_page_id': 'heapanalytics'
    },
    {
        'name': 'Pendo',
        'landing_url': 'https://www.pendo.io',
        'pricing_url': 'https://www.pendo.io/pricing',
        'g2_slug': 'pendo',
        'meta_page_id': 'pendoio'
    }
]

TARGET_AUDIENCE = 'Product Manager / Growth'

print(f'✅ {len(COMPETITORS)} competitors loaded')
for c in COMPETITORS:
    print(f'   • {c["name"]}')

---
## 🌐 CELL 4 — Layer 1A: Scrape landing page (real-time)

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def scrape_landing(url: str) -> dict:
    try:
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')

        headline = ''
        for sel in ['h1', '[class*=hero] h1', '[class*=headline]',
                    '[class*=banner] h1', 'main h1']:
            tag = soup.select_one(sel)
            if tag and tag.get_text(strip=True):
                headline = tag.get_text(strip=True)
                break

        subheadline = ''
        for sel in ['[class*=hero] p', '[class*=subtitle]',
                    '[class*=subhead]', 'header p', 'main p']:
            tag = soup.select_one(sel)
            txt = tag.get_text(strip=True) if tag else ''
            if txt and len(txt) > 20:
                subheadline = txt[:250]
                break

        cta = ''
        for sel in ['[class*=cta]', '[class*=btn-primary]',
                    '[class*=button-primary]', 'header a[class*=btn]',
                    'a[class*=primary]', 'button[class*=primary]']:
            tag = soup.select_one(sel)
            if tag and tag.get_text(strip=True):
                cta = tag.get_text(strip=True)
                break

        return {
            'headline': headline,
            'subheadline': subheadline,
            'cta': cta
        }
    except Exception as e:
        print(f'    ✗ Landing failed: {e}')
        return {'headline': '', 'subheadline': '', 'cta': ''}

# Quick test
print('Testing Mixpanel...')
t = scrape_landing('https://mixpanel.com')
print(f'  Headline : {t["headline"][:80]}')
print(f'  Sub      : {t["subheadline"][:80]}')
print(f'  CTA      : {t["cta"]}')

---
## 💰 CELL 5 — Layer 1B: Scrape pricing (real-time)

In [ ]:
import re

def scrape_pricing(url: str) -> dict:
    try:
        res = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')
        html = res.text

        # Plan names
        plans = []
        for sel in ['[class*=plan] h2', '[class*=tier] h3',
                    '[class*=plan] h3', '[class*=pricing] h2',
                    '[class*=plan-name]', '[class*=plan-title]']:
            tags = soup.select(sel)
            if tags:
                plans = [t.get_text(strip=True) for t in tags[:6]
                         if t.get_text(strip=True)]
                if plans: break

        # Prices — USD patterns
        prices = re.findall(r'\$[\d,]+(?:\.\d{2})?(?:/mo(?:nth)?)?', html)
        prices = list(dict.fromkeys(prices))[:8]

        # Free tier
        has_free = bool(re.search(r'\bfree\b', html, re.IGNORECASE))

        # Pricing model: PLG vs Sales-led
        plg_signals   = len(re.findall(
            r'start free|free trial|sign up free|get started free', html, re.IGNORECASE))
        sales_signals = len(re.findall(
            r'contact sales|talk to sales|request demo|book a demo', html, re.IGNORECASE))
        if plg_signals > sales_signals:
            pricing_model = 'PLG'
        elif sales_signals > plg_signals:
            pricing_model = 'Sales-led'
        else:
            pricing_model = 'Hybrid'

        return {
            'plans': plans,
            'prices': prices,
            'has_free_tier': has_free,
            'pricing_model': pricing_model
        }
    except Exception as e:
        print(f'    ✗ Pricing failed: {e}')
        return {'plans': [], 'prices': [], 'has_free_tier': False, 'pricing_model': 'Unknown'}

print('Testing Amplitude pricing...')
t = scrape_pricing('https://amplitude.com/pricing')
print(f'  Plans  : {t["plans"]}')
print(f'  Prices : {t["prices"]}')
print(f'  Model  : {t["pricing_model"]}')

---
## ⭐ CELL 6 — Layer 1C: G2 reviews (real-time)
> Để trống G2_API_KEY nếu chưa có — sẽ dùng public scrape

In [ ]:
G2_API_KEY = ''  # ← điền nếu có

def get_g2_data(slug: str) -> dict:
    if G2_API_KEY:
        try:
            url = f'https://data.g2.com/api/v1/products/{slug}/reviews'
            res = requests.get(
                url,
                headers={'Authorization': f'Token {G2_API_KEY}'},
                timeout=10
            )
            data = res.json()
            reviews = data.get('data', [])
            ratings = [r['attributes']['rating'] for r in reviews
                       if 'rating' in r.get('attributes', {})]
            avg = round(sum(ratings)/len(ratings), 1) if ratings else 0
            return {'avg_rating': avg, 'review_count': len(reviews), 'source': 'api'}
        except Exception as e:
            print(f'    G2 API error: {e}')

    # Fallback: public scrape
    try:
        url  = f'https://www.g2.com/products/{slug}/reviews'
        res  = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')

        rating_tag = soup.select_one("[itemprop='ratingValue']")
        count_tag  = soup.select_one("[itemprop='reviewCount']")

        return {
            'avg_rating'  : float(rating_tag['content']) if rating_tag else 0,
            'review_count': int(count_tag['content'])    if count_tag  else 0,
            'source'      : 'scrape'
        }
    except Exception as e:
        print(f'    ✗ G2 failed: {e}')
        return {'avg_rating': 0, 'review_count': 0, 'source': 'failed'}

print('Testing G2 PostHog...')
t = get_g2_data('posthog')
print(f'  Rating : {t["avg_rating"]}⭐ ({t["review_count"]} reviews) via {t["source"]}')

---
## 📣 CELL 7 — Layer 1D: Meta Ad Library (real-time)
> Lấy số ads đang chạy và messages từ Meta Ad Library (public, không cần login)

In [ ]:
def scrape_meta_ads(page_id: str, company_name: str) -> dict:
    """
    Scrape Meta Ad Library — public endpoint
    Trả về số ads đang active và sample messages
    """
    try:
        # Meta Ad Library public search URL
        url = (
            f'https://www.facebook.com/ads/library/?'
            f'active_status=active&ad_type=all&country=US'
            f'&q={company_name}&search_type=keyword_unordered'
        )
        res = requests.get(url, headers=HEADERS, timeout=15)

        # Extract ad count từ meta tags
        soup = BeautifulSoup(res.text, 'html.parser')

        # Tìm số ads trong response text
        ad_count_match = re.search(
            r'(\d+(?:,\d+)?)\s+(?:results?|ads?)', res.text, re.IGNORECASE)
        ad_count = 0
        if ad_count_match:
            ad_count = int(ad_count_match.group(1).replace(',', ''))

        # Extract sample ad messages từ JSON trong page source
        messages = []
        json_matches = re.findall(r'"body":\s*\{"markup":\{"__html":"([^"]{20,200})"', res.text)
        for m in json_matches[:5]:
            clean = re.sub(r'<[^>]+>', '', m)  # strip HTML tags
            clean = clean.replace('\\n', ' ').replace('\\"', '"').strip()
            if clean and len(clean) > 15:
                messages.append(clean[:150])

        # Detect ad themes
        all_text = ' '.join(messages).lower()
        themes = []
        theme_keywords = {
            'free trial'  : ['free', 'trial', 'start free'],
            'ROI/results' : ['roi', 'revenue', 'growth', 'increase'],
            'ease of use' : ['easy', 'simple', 'no-code', 'minutes'],
            'enterprise'  : ['enterprise', 'scale', 'team', 'company'],
            'technical'   : ['api', 'sdk', 'developer', 'open source'],
        }
        for theme, keywords in theme_keywords.items():
            if any(k in all_text for k in keywords):
                themes.append(theme)

        return {
            'ad_count'    : ad_count,
            'ad_messages' : messages,
            'ad_themes'   : themes
        }

    except Exception as e:
        print(f'    ✗ Meta Ads failed: {e}')
        return {'ad_count': 0, 'ad_messages': [], 'ad_themes': []}

print('Testing Meta Ads - Mixpanel...')
t = scrape_meta_ads('mixpanel', 'Mixpanel')
print(f'  Active ads : {t["ad_count"]}')
print(f'  Themes     : {t["ad_themes"]}')
print(f'  Sample msg : {t["ad_messages"][:2]}')

---
## 🏆 CELL 8 — Layer 3: Scoring Engine (6 tiêu chí)
Mỗi tiêu chí 0–10 điểm dựa trên rule cụ thể

In [ ]:
def score_competitor(data: dict) -> dict:
    """
    Chấm điểm 6 tiêu chí, mỗi tiêu chí 0-10
    Target audience: Product Manager / Growth
    """
    scores = {}
    notes  = {}
    headline    = data.get('headline', '').lower()
    subheadline = data.get('subheadline', '').lower()
    cta         = data.get('cta', '').lower()
    full_text   = f'{headline} {subheadline} {cta}'

    # ── 1. CLARITY (0-10) ────────────────────────────────────
    # Headline có nói rõ sản phẩm làm gì không?
    clarity_keywords = [
        'analytics', 'track', 'data', 'insight', 'measure',
        'understand', 'behavior', 'product', 'user', 'event'
    ]
    keyword_hits = sum(1 for k in clarity_keywords if k in headline)
    headline_len = len(headline.split())

    clarity = min(10, keyword_hits * 2)
    if 5 <= headline_len <= 12:  # sweet spot length
        clarity += 2
    clarity = min(10, clarity)
    scores['clarity'] = clarity
    notes['clarity']  = f'{keyword_hits} relevant keywords in headline ({headline_len} words)'

    # ── 2. VALUE PROP (0-10) ─────────────────────────────────
    # Benefit cụ thể hay chỉ feature?
    benefit_keywords = [
        'faster', 'save', 'increase', 'reduce', 'revenue',
        'growth', 'convert', 'retain', 'improve', 'boost',
        'minutes', 'instantly', 'without', 'easier'
    ]
    feature_keywords = [
        'platform', 'tool', 'software', 'solution', 'system'
    ]
    benefit_hits = sum(1 for k in benefit_keywords if k in full_text)
    feature_hits = sum(1 for k in feature_keywords if k in full_text)

    value_prop = min(10, benefit_hits * 2 - feature_hits)
    value_prop = max(0, value_prop)
    scores['value_prop'] = value_prop
    notes['value_prop']  = f'{benefit_hits} benefit signals, {feature_hits} generic feature words'

    # ── 3. CTA STRENGTH (0-10) ───────────────────────────────
    strong_cta  = ['start', 'get', 'try', 'build', 'launch', 'create', 'join']
    weak_cta    = ['learn more', 'explore', 'see', 'discover', 'view']
    free_signal = ['free', 'trial', 'demo']

    cta_score = 4  # baseline
    if any(k in cta for k in strong_cta):
        cta_score += 3
    if any(k in cta for k in weak_cta):
        cta_score -= 2
    if any(k in cta for k in free_signal):
        cta_score += 2
    if len(cta.split()) <= 4:
        cta_score += 1  # concise CTA
    cta_score = max(0, min(10, cta_score))
    scores['cta_strength'] = cta_score
    notes['cta_strength']  = f'CTA: "{data.get("cta", "N/A")}"'

    # ── 4. SOCIAL PROOF (0-10) ───────────────────────────────
    rating = data.get('avg_rating', 0)
    count  = data.get('review_count', 0)

    social = 0
    if rating >= 4.5: social += 5
    elif rating >= 4.0: social += 3
    elif rating >= 3.5: social += 1

    if count >= 10000: social += 5
    elif count >= 5000: social += 4
    elif count >= 1000: social += 2
    elif count >= 100:  social += 1

    social = min(10, social)
    scores['social_proof'] = social
    notes['social_proof']  = f'G2: {rating}⭐ · {count:,} reviews'

    # ── 5. PRICING TRANSPARENCY (0-10) ───────────────────────
    pricing_score = 0
    if data.get('prices'):           pricing_score += 4  # có giá public
    if data.get('has_free_tier'):    pricing_score += 3  # có free tier
    if data.get('plans'):            pricing_score += 2  # có plan names rõ
    if data.get('pricing_model') == 'PLG': pricing_score += 1  # PLG signal
    pricing_score = min(10, pricing_score)
    scores['pricing_transparency'] = pricing_score
    notes['pricing_transparency']  = (
        f'Model: {data.get("pricing_model", "?")}, '
        f'Free: {data.get("has_free_tier")}, '
        f'Plans: {len(data.get("plans", []))}'
    )

    # ── 6. AUDIENCE CLARITY (0-10) ───────────────────────────
    # Nhắm vào PM/Growth team không?
    pm_keywords = [
        'product', 'pm', 'product manager', 'growth', 'team',
        'product team', 'feature', 'retention', 'activation',
        'funnel', 'experiment', 'a/b test'
    ]
    pm_hits = sum(1 for k in pm_keywords if k in full_text)
    audience_score = min(10, pm_hits * 2)
    scores['audience_clarity'] = audience_score
    notes['audience_clarity']  = f'{pm_hits} PM/Growth signals in copy'

    # ── TOTAL ────────────────────────────────────────────────
    total = round(sum(scores.values()) / len(scores), 1)

    return {
        'total' : total,
        'scores': scores,
        'notes' : notes
    }

# Quick test với data giả
test_data = {
    'headline'      : 'Build better products with product analytics',
    'subheadline'   : 'Understand user behavior and improve retention',
    'cta'           : 'Start for free',
    'avg_rating'    : 4.5,
    'review_count'  : 8000,
    'has_free_tier' : True,
    'plans'         : ['Free', 'Growth', 'Enterprise'],
    'prices'        : ['$0', '$49/mo'],
    'pricing_model' : 'PLG'
}
result = score_competitor(test_data)
print(f'Test score: {result["total"]}/10')
for k, v in result['scores'].items():
    print(f'  {k:<25}: {v}/10  — {result["notes"][k]}')

---
## 🔐 CELL 9 — Layer 2: Hash engine + Save snapshot

In [ ]:
def hash_fields(data: dict) -> dict:
    return {
        f: hashlib.md5(str(v).encode()).hexdigest()
        for f, v in data.items()
    }

def save_snapshot(competitor: str, data: dict, score: dict):
    conn = sqlite3.connect(DB_PATH)

    hashes = hash_fields({
        'headline'   : data.get('headline', ''),
        'cta'        : data.get('cta', ''),
        'prices'     : str(data.get('prices', [])),
        'plans'      : str(data.get('plans', [])),
        'ad_count'   : str(data.get('ad_count', 0))
    })

    conn.execute("""
        INSERT INTO snapshots
        (date, competitor, headline, subheadline, cta,
         plans, prices, has_free, pricing_model,
         g2_rating, g2_count, ad_count, ad_messages,
         score_total, score_detail, hashes_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        date.today().isoformat(), competitor,
        data.get('headline', ''), data.get('subheadline', ''),
        data.get('cta', ''),
        json.dumps(data.get('plans', [])),
        json.dumps(data.get('prices', [])),
        int(data.get('has_free_tier', False)),
        data.get('pricing_model', ''),
        data.get('avg_rating', 0), data.get('review_count', 0),
        data.get('ad_count', 0),
        json.dumps(data.get('ad_messages', [])),
        score.get('total', 0),
        json.dumps(score),
        json.dumps(hashes)
    ))
    conn.commit()
    conn.close()

def detect_changes(competitor: str) -> list:
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("""
        SELECT date, headline, cta, prices, plans, ad_count, hashes_json
        FROM snapshots WHERE competitor=?
        ORDER BY date DESC LIMIT 2
    """, (competitor,)).fetchall()
    conn.close()

    if len(rows) < 2:
        return []

    new, old = rows[0], rows[1]
    new_h = json.loads(new[6])
    old_h = json.loads(old[6])

    field_map = {
        'headline' : (new[1], old[1]),
        'cta'      : (new[2], old[2]),
        'prices'   : (new[3], old[3]),
        'plans'    : (new[4], old[4]),
        'ad_count' : (str(new[5]), str(old[5]))
    }

    changes = []
    for field, (new_val, old_val) in field_map.items():
        if new_h.get(field) != old_h.get(field):
            changes.append({
                'field' : field,
                'before': old_val,
                'after' : new_val,
                'date'  : new[0]
            })
    return changes

print('✅ Hash engine + snapshot functions ready')

---
## 🚀 CELL 10 — Chạy toàn bộ pipeline (real-time)
> ⏱️ Mỗi lần chạy cell này = 1 weekly snapshot mới

In [ ]:
def run_pipeline():
    print('=' * 60)
    print('🚀 COMPETITOR INTELLIGENCE PIPELINE — REAL-TIME')
    print(f'   Date: {date.today().isoformat()}')
    print('=' * 60)

    all_results = []

    for c in COMPETITORS:
        print(f'\n🔍 [{c["name"]}]')

        # Layer 1A — Landing
        landing = scrape_landing(c['landing_url'])
        icon = '✓' if landing['headline'] else '✗'
        print(f'  {icon} Landing  → "{landing["headline"][:65]}"')
        time.sleep(2)

        # Layer 1B — Pricing
        pricing = scrape_pricing(c['pricing_url'])
        print(f'  ✓ Pricing  → {pricing["pricing_model"]} | plans: {pricing["plans"]}')
        time.sleep(2)

        # Layer 1C — G2
        reviews = get_g2_data(c['g2_slug'])
        print(f'  ✓ G2       → {reviews["avg_rating"]}⭐ ({reviews["review_count"]:,} reviews)')
        time.sleep(2)

        # Layer 1D — Meta Ads
        ads = scrape_meta_ads(c['meta_page_id'], c['name'])
        print(f'  ✓ Meta Ads → {ads["ad_count"]} active ads | themes: {ads["ad_themes"]}')
        time.sleep(2)

        # Merge all data
        combined = {**landing, **pricing, **reviews, **ads}

        # Layer 3 — Score
        score = score_competitor(combined)
        print(f'  ✓ Score    → {score["total"]}/10')
        for criterion, s in score['scores'].items():
            bar = '█' * s + '░' * (10-s)
            print(f'      {criterion:<25} {bar} {s}/10')

        # Layer 2 — Save
        save_snapshot(c['name'], combined, score)
        print(f'  ✓ Saved to DB')

        # Detect changes
        changes = detect_changes(c['name'])
        if changes:
            print(f'  🔴 {len(changes)} change(s) detected:')
            for ch in changes:
                print(f'     [{ch["field"]}]: "{str(ch["before"])[:40]}" → "{str(ch["after"])[:40]}"')
        else:
            print(f'  ✅ No changes vs last snapshot')

        all_results.append({
            'competitor': c['name'],
            'data'      : combined,
            'score'     : score,
            'changes'   : changes
        })
        time.sleep(3)

    print('\n' + '='*60)
    print('✅ PIPELINE COMPLETE')
    print('='*60)
    return all_results

results = run_pipeline()

---
## 📊 CELL 11 — Leaderboard + Insights

In [ ]:
import pandas as pd

def print_leaderboard(results):
    print(f'\n🏆 SCORING LEADERBOARD — {date.today().isoformat()}')
    print('='*60)

    # Sort by score
    ranked = sorted(results, key=lambda x: x['score']['total'], reverse=True)

    for i, r in enumerate(ranked, 1):
        score  = r['score']['total']
        bar    = '█' * int(score) + '░' * (10 - int(score))
        medal  = ['🥇','🥈','🥉','4️⃣','5️⃣'][i-1]
        change_flag = '🔴' if r['changes'] else '✅'
        print(f'{medal} {r["competitor"]:<15} {bar} {score}/10  {change_flag}')

    print()

    # Breakdown table
    print('SCORE BREAKDOWN')
    print('-'*60)
    criteria = ['clarity','value_prop','cta_strength','social_proof',
                'pricing_transparency','audience_clarity']
    header = f'{"":<15}' + ''.join(f'{c[:6]:>8}' for c in criteria) + f'{"TOTAL":>8}'
    print(header)
    print('-'*60)
    for r in ranked:
        row = f'{r["competitor"]:<15}'
        for c in criteria:
            row += f'{r["score"]["scores"].get(c, 0):>8}'
        row += f'{r["score"]["total"]:>8}'
        print(row)

    # Key insights
    print('\nKEY INSIGHTS')
    print('-'*60)
    top    = ranked[0]
    bottom = ranked[-1]
    print(f'→ Best overall   : {top["competitor"]} ({top["score"]["total"]}/10)')
    print(f'→ Weakest overall: {bottom["competitor"]} ({bottom["score"]["total"]}/10)')

    # Find best per criterion
    for c in criteria:
        best = max(results, key=lambda x: x['score']['scores'].get(c, 0))
        print(f'→ Best {c:<22}: {best["competitor"]} ({best["score"]["scores"].get(c,0)}/10)')

print_leaderboard(results)

---
## 🤖 CELL 12 — A/B Generator (rule-based)
Generate 2 variants dựa trên competitor scores — không cần API key

In [ ]:
YOUR_PRODUCT_NAME = 'YourProduct'   # ← đổi tên sản phẩm của bạn
YOUR_USP          = 'the fastest way to understand why users drop off'  # ← USP của bạn

def generate_ab_variants(results, product_name, usp):
    ranked = sorted(results, key=lambda x: x['score']['total'], reverse=True)
    top_comp = ranked[0]

    print('\n🧪 A/B TEST GENERATOR')
    print('='*60)
    print(f'Target audience: Product Manager / Growth')
    print(f'Your product   : {product_name}')
    print(f'Your USP       : {usp}')

    # ── VARIANT A: Follow the leader ──────────────────────────
    print(f'\n── VARIANT A: Follow the Leader ──')
    print(f'Based on: {top_comp["competitor"]} (top score: {top_comp["score"]["total"]}/10)')
    print(f'Original headline: "{top_comp["data"].get("headline", "")}"')
    print(f'Original CTA     : "{top_comp["data"].get("cta", "")}"')
    print()

    # Adapt structure của top competitor cho product mình
    orig_h = top_comp['data'].get('headline', '')
    # Pattern: giữ structure, thay nội dung bằng USP của mình
    if 'better' in orig_h.lower():
        variant_a_h = f'Build better products with {product_name}'
    elif 'understand' in orig_h.lower() or 'insight' in orig_h.lower():
        variant_a_h = f'Understand your users instantly with {product_name}'
    elif 'data' in orig_h.lower():
        variant_a_h = f'Turn product data into decisions with {product_name}'
    else:
        variant_a_h = f'The analytics platform built for product teams'

    variant_a_sub = f'Join product teams who use {product_name} to {usp}'
    variant_a_cta = top_comp['data'].get('cta', 'Start for free')

    print(f'→ Headline : "{variant_a_h}"')
    print(f'→ Sub      : "{variant_a_sub}"')
    print(f'→ CTA      : "{variant_a_cta}"')
    print(f'→ Logic    : Sao chép structure của {top_comp["competitor"]} — đã proven convert tốt')

    # ── VARIANT B: Contrarian ─────────────────────────────────
    print(f'\n── VARIANT B: Contrarian Positioning ──')

    # Tìm pattern lặp lại của tất cả competitors
    all_headlines = [r['data'].get('headline', '').lower() for r in results]
    all_ctas      = [r['data'].get('cta', '').lower() for r in results]
    common_words  = {}
    for h in all_headlines:
        for word in h.split():
            if len(word) > 4:
                common_words[word] = common_words.get(word, 0) + 1
    overused = [w for w, c in common_words.items() if c >= 2]

    print(f'Overused words across all competitors: {overused[:8]}')
    print()

    # Generate contrarian berdasarkan apa yang TIDAK ada
    has_free_majority = sum(1 for r in results if r['data'].get('has_free_tier')) > len(results)/2
    plg_majority = sum(1 for r in results
                       if r['data'].get('pricing_model') == 'PLG') > len(results)/2

    if 'analytics' in overused or 'data' in overused:
        variant_b_h = f'Stop analyzing. Start deciding. — {product_name}'
    elif 'product' in overused:
        variant_b_h = f'Your users are telling you something. {product_name} translates it.'
    else:
        variant_b_h = f'While they show you charts, {product_name} shows you answers.'

    if has_free_majority:
        # Mọi người đều free → đi ngược: emphasize kết quả thay vì free
        variant_b_sub = f'Not another free dashboard. {usp.capitalize()} — in 10 minutes.'
        variant_b_cta = 'See your first insight now'
    else:
        variant_b_sub = f'Free to start, {usp}. No SQL required.'
        variant_b_cta = 'Start free — no credit card'

    print(f'→ Headline : "{variant_b_h}"')
    print(f'→ Sub      : "{variant_b_sub}"')
    print(f'→ CTA      : "{variant_b_cta}"')
    print(f'→ Logic    : Mọi competitor nói "{overused[:3]}" → bạn ZAG sang outcomes')

    return {
        'variant_a': {'headline': variant_a_h, 'sub': variant_a_sub, 'cta': variant_a_cta},
        'variant_b': {'headline': variant_b_h, 'sub': variant_b_sub, 'cta': variant_b_cta},
        'top_competitor': top_comp['competitor'],
        'overused_words': overused[:8]
    }

ab_variants = generate_ab_variants(results, YOUR_PRODUCT_NAME, YOUR_USP)

---
## 📁 CELL 13 — Export data cho Streamlit app

In [ ]:
def export_for_app(results, ab_variants):
    """
    Export toàn bộ data ra JSON để Streamlit app đọc
    File này replace mock_data.json cũ
    """
    snapshots_export = []
    conn = sqlite3.connect(DB_PATH)

    for c in COMPETITORS:
        rows = conn.execute("""
            SELECT date, headline, subheadline, cta, plans, prices,
                   has_free, pricing_model, g2_rating, g2_count,
                   ad_count, ad_messages, score_total, score_detail
            FROM snapshots WHERE competitor=?
            ORDER BY date ASC
        """, (c['name'],)).fetchall()

        for row in rows:
            score_detail = json.loads(row[13]) if row[13] else {}
            snapshots_export.append({
                'date'          : row[0],
                'competitor'    : c['name'],
                'headline'      : row[1],
                'subheadline'   : row[2],
                'cta'           : row[3],
                'plans'         : json.loads(row[4]) if row[4] else [],
                'prices'        : json.loads(row[5]) if row[5] else [],
                'has_free_tier' : bool(row[6]),
                'pricing_model' : row[7],
                'avg_rating'    : row[8],
                'review_count'  : row[9],
                'ad_count'      : row[10],
                'ad_messages'   : json.loads(row[11]) if row[11] else [],
                'score_total'   : row[12],
                'score_detail'  : score_detail
            })
    conn.close()

    output = {
        'last_updated': date.today().isoformat(),
        'competitors' : [c['name'] for c in COMPETITORS],
        'snapshots'   : snapshots_export,
        'ab_variants' : ab_variants
    }

    out_path = f'{BASE_DIR}/live_data.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f'✅ Exported {len(snapshots_export)} snapshots → {out_path}')
    print(f'   Upload file này lên GitHub repo để Streamlit app tự cập nhật')
    return out_path

export_path = export_for_app(results, ab_variants)

# Download về máy
from google.colab import files
files.download(export_path)

---
## 📝 Ghi chú sử dụng

### Chạy lần đầu
1. Cell 1 → 3: setup
2. Cell 4 → 7: test từng scraper
3. Cell 8: xem scoring logic
4. Cell 10: chạy toàn bộ pipeline
5. Cell 11 → 12: xem leaderboard + A/B variants
6. Cell 13: export `live_data.json` → upload lên GitHub

### Mỗi tuần
- Chỉ cần chạy **Cell 10** + **Cell 13**
- Upload `live_data.json` lên GitHub → Streamlit tự cập nhật

### Tuỳ chỉnh A/B Generator (Cell 12)
- Đổi `YOUR_PRODUCT_NAME` thành tên sản phẩm thực tế
- Đổi `YOUR_USP` thành điểm khác biệt của bạn

### Bước tiếp theo
- Update Streamlit app để đọc `live_data.json` thay vì `mock_data.json`
- Thêm tab A/B Generator vào app
- Thêm Anthropic API key để AI generate variants thông minh hơn